## This notebook compiles crispresso results and counts of validation screen.  

Kexin Dong

Nov 25, 2025

In [1]:
import numpy as np 
import pandas as pd
import os
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']

In [2]:
LIB = pd.read_csv('/Users/kexindong/Documents/GitHub/PhD-FSR-MH-Lab/07_B-ALL_resubmission_20250819/validation_screen_analysis/STEP3_generate_config_file/FINAL_focused_library.csv')
ABE_list = list(LIB[LIB['Editor']=='ABE']['gRNA_id'])
CBE_list = list(LIB[LIB['Editor']=='ABE']['gRNA_id'])

In [3]:
def crispresso_compiler(samp, sample_id, fp):
    """ 
    Takes in list of sample names (see cell above)
    Returns compiled dictionary of dataframes containing editing information for each sensor
    - key for each is the sample_id name
    
    Includes:
    1. corr_perc = pure correct editing
    2. target_base_edit_perc = target base editing perc (including edits with bystander editing)
    3. wt_perc
    4. byproduct information (indels, substitutions, ambiguous)
    """

    df_edits = []
    for k in samp: 
        concated = pd.read_csv(f'{fp}/{k}_crispresso_aggregated.csv')
        concated = concated.fillna(0)

        #somehow wasn't able to find the initial set of code that I used for this...

        #go through all of the samples and do it step by step
        sample_ids = []
        #sample_num = []
        rii = []
        raaa = []
        r_lowqual = []
        ra_hdr = []
        ra_wt = []
        r_unaligned = []
        no_edit = []
        correct = []
        target_base_editing = []
        byproduct_all = []
        byproduct_indel = []
        byproduct_sub = []
        byproduct_ambig = []
        for i in np.unique(concated['Guide_ID']):
            sample_ids.append(i)

            subset = concated[concated['Guide_ID']==i]

            wt = subset[subset['Amplicon']=='Reference']
            edit = subset[subset['Amplicon']=='HDR']

            #sample_num.append(wt['sample_num'].values[0])
            rii1 = wt['Reads_in_input'].values[0]
            r1 = wt['Reads_aligned_all_amplicons'].values[0]
            rii.append(rii1)
            raaa.append(r1)
            r_lowqual.append(rii1-r1)

            r2 = edit['Reads_aligned'].values[0]
            r3 = wt['Reads_aligned'].values[0]
            ra_hdr.append(r2)
            ra_wt.append(r3)
            r_unaligned.append(r1 - (r2+r3))

            no_edit.append(wt['Unmodified'].values[0])
            correct.append(edit['Unmodified'].values[0])
            target_base_editing.append(edit['Modified'].values[0] + edit['Unmodified'].values[0])

            byprod_all = wt['Modified'].values[0] + edit['Modified'].values[0] + (r1 - (r2+r3)) #add unaligned reads
            sub_all = wt['Only Substitutions'].values[0] + edit['Only Substitutions'].values[0]
            indel_all = wt['Only Deletions'].values[0] + wt['Only Insertions'].values[0] + wt['Insertions and Deletions'].values[0] + edit['Only Deletions'].values[0] + edit['Only Insertions'].values[0] + edit['Insertions and Deletions'].values[0]
            ambig_all = byprod_all - sub_all - indel_all

            byproduct_all.append(byprod_all)
            byproduct_indel.append(indel_all)
            byproduct_sub.append(sub_all)
            byproduct_ambig.append(ambig_all)


        cols = ["Guide_ID", "Reads_in_input","Reads_lowqual", "Reads_aligned_all_amplicons","Reads_aligned_WT", "Reads_aligned_HDR", "Reads_unaligned", "WT","correct_edit", "target_base_edit", "byproduct_all","byproduct_INDEL","byproduct_sub","byproduct_ambiguous"]
        col_vals = [ sample_ids, rii, r_lowqual, raaa, ra_wt, ra_hdr, r_unaligned, no_edit, correct, target_base_editing, byproduct_all, byproduct_indel, byproduct_sub, byproduct_ambig]

        out = pd.DataFrame(dict(zip(cols, col_vals)))
        out['corr_perc'] = 100*(out['correct_edit']/out['Reads_aligned_all_amplicons'])
        out['target_base_edit_perc'] = 100*(out['target_base_edit']/out['Reads_aligned_all_amplicons'])
        out['WT_perc'] = 100*(out['WT']/out['Reads_aligned_all_amplicons'])
        out['byproduct_all_perc'] =  100*(out['byproduct_all']/out['Reads_aligned_all_amplicons'])
        out['byproduct_INDEL_perc'] =  100*(out['byproduct_INDEL']/out['Reads_aligned_all_amplicons'])
        out['byproduct_sub_perc'] =  100*(out['byproduct_sub']/out['Reads_aligned_all_amplicons'])
        out['byproduct_ambiguous_perc'] =100*(out['byproduct_ambiguous']/out['Reads_aligned_all_amplicons'])
        out = out.fillna(0)
        out = out.sort_index()
        df_edits.append(out)

    edit_dict = dict(zip(sample_id, df_edits))

    return edit_dict

In [4]:
config = pd.read_csv('CONFIG_BALL_VALIDATION_SCREEN.txt', sep=' ')
list(config['folder_name'])

['D25-13245-1-7331E_guide_split_BARCODES',
 'D25-13245-2-7331E_guide_split_BARCODES',
 'D25-13246-1-7331E_guide_split_BARCODES',
 'D25-13246-2-7331E_guide_split_BARCODES',
 'D25-13247-1-7331E_guide_split_BARCODES',
 'D25-13247-2-7331E_guide_split_BARCODES',
 'D25-13248-1-7331E_guide_split_BARCODES',
 'D25-13248-2-7331E_guide_split_BARCODES',
 'D25-13249-1-7331E_guide_split_BARCODES',
 'D25-13249-2-7331E_guide_split_BARCODES',
 'D25-13250-1-7331E_guide_split_BARCODES',
 'D25-13250-2-7331E_guide_split_BARCODES',
 'D25-13251-1-7331E_guide_split_BARCODES',
 'D25-13251-2-7331E_guide_split_BARCODES',
 'D25-13252-1-7331E_guide_split_BARCODES',
 'D25-13252-2-7331E_guide_split_BARCODES',
 'D25-13253-1-7331E_guide_split_BARCODES',
 'D25-13253-2-7331E_guide_split_BARCODES',
 'D25-13254-1-7331E_guide_split_BARCODES',
 'D25-13254-2-7331E_guide_split_BARCODES',
 'D25-13255-1-7331E_guide_split_BARCODES',
 'D25-13255-2-7331E_guide_split_BARCODES',
 'D25-13256-1-7331E_guide_split_BARCODES',
 'D25-13256

In [5]:
id_in_vivo = []
for y in range(5):
    for x in ['spleen','bm','men']:
        for z in range(2):
            id_in_vivo.append(f'{x}{y+1}-{z+1}')
id_in_vivo

['spleen1-1',
 'spleen1-2',
 'bm1-1',
 'bm1-2',
 'men1-1',
 'men1-2',
 'spleen2-1',
 'spleen2-2',
 'bm2-1',
 'bm2-2',
 'men2-1',
 'men2-2',
 'spleen3-1',
 'spleen3-2',
 'bm3-1',
 'bm3-2',
 'men3-1',
 'men3-2',
 'spleen4-1',
 'spleen4-2',
 'bm4-1',
 'bm4-2',
 'men4-1',
 'men4-2',
 'spleen5-1',
 'spleen5-2',
 'bm5-1',
 'bm5-2',
 'men5-1',
 'men5-2']

### ABE BC

In [6]:
samp_ABE_BC = [
    'D25-13337-1-7331E','D25-13337-2-7331E', #library

    'D25-13245-1-7331E','D25-13245-2-7331E', #input

    'D25-13251-1-7331E','D25-13251-2-7331E', #in vitro d5

    'D25-13317-1-7331E','D25-13317-2-7331E', #in vitro d15
    'D25-13788-1-7331E','D25-13788-2-7331E',
    'D25-13789-1-7331E','D25-13789-2-7331E',
    'D25-13790-1-7331E','D25-13790-2-7331E',
    'D25-13321-1-7331E','D25-13321-2-7331E',

    'D25-13278-1-7331E','D25-13278-2-7331E', #spleen1
    'D25-13279-1-7331E','D25-13279-2-7331E', #bm1
    'D25-13280-1-7331E','D25-13280-2-7331E', #men1
    'D25-13281-1-7331E','D25-13281-2-7331E',
    'D25-13282-1-7331E','D25-13282-2-7331E',
    'D25-13283-1-7331E','D25-13283-2-7331E',
    'D25-13284-1-7331E','D25-13284-2-7331E',
    'D25-13285-1-7331E','D25-13285-2-7331E',
    'D25-13286-1-7331E','D25-13286-2-7331E',
    'D25-13287-1-7331E','D25-13287-2-7331E',
    'D25-13288-1-7331E','D25-13288-2-7331E',
    'D25-13289-1-7331E','D25-13289-2-7331E',
    'D25-13290-1-7331E','D25-13290-2-7331E',
    'D25-13291-1-7331E','D25-13291-2-7331E',
    'D25-13292-1-7331E','D25-13292-2-7331E',
]
samp_ABE_BC = [f'{x}_guide_split_BARCODES' for x in samp_ABE_BC]
id_abe_bc_d15 = []
for y in range(5):
    for z in range(2):
        id_abe_bc_d15.append(f'd15-rep{y+1}-{z+1}')
id_abe_bc_d15

samp_id_ABE_BC = [
    'lib-1','lib-2',
    'input-1','input-2',
    'd5-1','d5-2',
] + id_abe_bc_d15 + id_in_vivo

In [7]:
samp_id_ABE_BC

['lib-1',
 'lib-2',
 'input-1',
 'input-2',
 'd5-1',
 'd5-2',
 'd15-rep1-1',
 'd15-rep1-2',
 'd15-rep2-1',
 'd15-rep2-2',
 'd15-rep3-1',
 'd15-rep3-2',
 'd15-rep4-1',
 'd15-rep4-2',
 'd15-rep5-1',
 'd15-rep5-2',
 'spleen1-1',
 'spleen1-2',
 'bm1-1',
 'bm1-2',
 'men1-1',
 'men1-2',
 'spleen2-1',
 'spleen2-2',
 'bm2-1',
 'bm2-2',
 'men2-1',
 'men2-2',
 'spleen3-1',
 'spleen3-2',
 'bm3-1',
 'bm3-2',
 'men3-1',
 'men3-2',
 'spleen4-1',
 'spleen4-2',
 'bm4-1',
 'bm4-2',
 'men4-1',
 'men4-2',
 'spleen5-1',
 'spleen5-2',
 'bm5-1',
 'bm5-2',
 'men5-1',
 'men5-2']

### ABE EPO

In [8]:
samp_ABE_EPO = [
    'D25-13337-1-7331E','D25-13337-2-7331E', #library

    'D25-13245-1-7331E','D25-13245-2-7331E', #input

    'D25-13257-1-7331E','D25-13257-2-7331E', #in vitro d5

    'D25-13322-1-7331E','D25-13322-2-7331E', #in vitro d15
    'D25-13791-1-7331E','D25-13791-2-7331E',
    'D25-13324-1-7331E','D25-13324-2-7331E',
    'D25-13325-1-7331E','D25-13325-2-7331E',
    'D25-13326-1-7331E','D25-13326-2-7331E',

    'D25-13263-1-7331E','D25-13263-2-7331E',#spleen1
    'D25-13264-1-7331E','D25-13264-2-7331E',#bm1
    'D25-13265-1-7331E','D25-13265-2-7331E',#men1
    'D25-13266-1-7331E','D25-13266-2-7331E',
    'D25-13267-1-7331E','D25-13267-2-7331E',
    'D25-13268-1-7331E','D25-13268-2-7331E',
    'D25-13269-1-7331E','D25-13269-2-7331E',
    'D25-13270-1-7331E','D25-13270-2-7331E',
    'D25-13271-1-7331E','D25-13271-2-7331E',
    'D25-13272-1-7331E','D25-13272-2-7331E',
    'D25-13273-1-7331E','D25-13273-2-7331E',
    'D25-13274-1-7331E','D25-13274-2-7331E',
    'D25-13275-1-7331E','D25-13275-2-7331E',
    'D25-13276-1-7331E','D25-13276-2-7331E',
    'D25-13277-1-7331E','D25-13277-2-7331E',

]
samp_ABE_EPO = [f'{x}_guide_split_BARCODES' for x in samp_ABE_EPO]
id_abe_bc_d15 = []
for y in range(5):
    for z in range(2):
        id_abe_bc_d15.append(f'd15-rep{y+1}-{z+1}')
id_abe_bc_d15

samp_id_ABE_EPO = [
    'lib-1','lib-2',
    'input-1','input-2',
    'd5-1','d5-2',
] + id_abe_bc_d15 + id_in_vivo

print(len(samp_ABE_EPO),len(samp_id_ABE_EPO))

46 46


In [10]:
samp_id_ABE_EPO

['lib-1',
 'lib-2',
 'input-1',
 'input-2',
 'd5-1',
 'd5-2',
 'd15-rep1-1',
 'd15-rep1-2',
 'd15-rep2-1',
 'd15-rep2-2',
 'd15-rep3-1',
 'd15-rep3-2',
 'd15-rep4-1',
 'd15-rep4-2',
 'd15-rep5-1',
 'd15-rep5-2',
 'spleen1-1',
 'spleen1-2',
 'bm1-1',
 'bm1-2',
 'men1-1',
 'men1-2',
 'spleen2-1',
 'spleen2-2',
 'bm2-1',
 'bm2-2',
 'men2-1',
 'men2-2',
 'spleen3-1',
 'spleen3-2',
 'bm3-1',
 'bm3-2',
 'men3-1',
 'men3-2',
 'spleen4-1',
 'spleen4-2',
 'bm4-1',
 'bm4-2',
 'men4-1',
 'men4-2',
 'spleen5-1',
 'spleen5-2',
 'bm5-1',
 'bm5-2',
 'men5-1',
 'men5-2']

### CBE BC

In [11]:
samp_CBE_BC = [
    'D25-13797-1-7331E','D25-13797-2-7331E', #library
    
    'D25-13246-1-7331E','D25-13246-2-7331E', #input rep1-5
    'D25-13247-1-7331E','D25-13247-2-7331E',
    'D25-13248-1-7331E','D25-13248-2-7331E',
    'D25-13249-1-7331E','D25-13249-2-7331E',
    'D25-13250-1-7331E','D25-13250-2-7331E',

    'D25-13252-1-7331E','D25-13252-2-7331E', #d5 rep1-5
    'D25-13253-1-7331E','D25-13253-2-7331E',
    'D25-13254-1-7331E','D25-13254-2-7331E',
    'D25-13255-1-7331E','D25-13255-2-7331E',
    'D25-13256-1-7331E','D25-13256-2-7331E',

    'D25-13792-1-7331E','D25-13792-2-7331E', #d15 rep1-5
    'D25-13328-1-7331E','D25-13328-2-7331E',
    'D25-13329-1-7331E','D25-13329-2-7331E',
    'D25-13330-1-7331E','D25-13330-2-7331E',
    'D25-13793-1-7331E','D25-13793-2-7331E',

    'D25-13308-1-7331E','D25-13308-2-7331E', #spleen1
    'D25-13309-1-7331E','D25-13309-2-7331E', #bm1
    'D25-13310-1-7331E','D25-13310-2-7331E', #men1
    'D25-13311-1-7331E','D25-13311-2-7331E', #spleen2
    'D25-13312-1-7331E','D25-13312-2-7331E', #bm2
    'D25-13313-1-7331E','D25-13313-2-7331E', #men2
    'D25-13314-1-7331E','D25-13314-2-7331E', #spleen4
    'D25-13315-1-7331E','D25-13315-2-7331E', #bm4
    'D25-13316-1-7331E','D25-13316-2-7331E', #men4
]
samp_CBE_BC = [f'{x}_guide_split_BARCODES' for x in samp_CBE_BC]
id_cbe = []
for x in ['input','d5','d15']:
    for y in range(5):
        for z in range(2):
            id_cbe.append(f'{x}-rep{y+1}-{z+1}')
id_cbe

in_vivo_id_1_2_4 = []
for y in [0,1,3]:
    for x in ['spleen','bm','men']:
        for z in range(2):
            in_vivo_id_1_2_4.append(f'{x}{y+1}-{z+1}')
id_in_vivo

samp_id_CBE_BC = ['lib-1','lib-2'] + id_cbe + in_vivo_id_1_2_4
print(len(samp_CBE_BC),len(samp_id_CBE_BC))

50 50


### CBE EPO

In [12]:
samp_CBE_EPO = [
    'D25-13797-1-7331E','D25-13797-2-7331E', #library
    
    'D25-13246-1-7331E','D25-13246-2-7331E', #input rep1-5
    'D25-13247-1-7331E','D25-13247-2-7331E',
    'D25-13248-1-7331E','D25-13248-2-7331E',
    'D25-13249-1-7331E','D25-13249-2-7331E',
    'D25-13250-1-7331E','D25-13250-2-7331E',

    'D25-13258-1-7331E','D25-13258-2-7331E', #d5 rep1-5
    'D25-13259-1-7331E','D25-13259-2-7331E',
    'D25-13260-1-7331E','D25-13260-2-7331E',
    'D25-13261-1-7331E','D25-13261-2-7331E',
    'D25-13262-1-7331E','D25-13262-2-7331E',

    'D25-13332-1-7331E','D25-13332-2-7331E', #d15 rep1-5
    'D25-13333-1-7331E','D25-13333-2-7331E', 
    'D25-13794-1-7331E','D25-13794-2-7331E',
    'D25-13795-1-7331E','D25-13795-2-7331E',
    'D25-13796-1-7331E','D25-13796-2-7331E',

    'D25-13293-1-7331E','D25-13293-2-7331E', #spleen1
    'D25-13294-1-7331E','D25-13294-2-7331E', #bm1
    'D25-13295-1-7331E','D25-13295-2-7331E', #men1
    'D25-13296-1-7331E','D25-13296-2-7331E',
    'D25-13297-1-7331E','D25-13297-2-7331E',
    'D25-13298-1-7331E','D25-13298-2-7331E',
    'D25-13299-1-7331E','D25-13299-2-7331E',
    'D25-13300-1-7331E','D25-13300-2-7331E',
    'D25-13301-1-7331E','D25-13301-2-7331E',
    'D25-13302-1-7331E','D25-13302-2-7331E',
    'D25-13303-1-7331E','D25-13303-2-7331E',
    'D25-13304-1-7331E','D25-13304-2-7331E',
    'D25-13305-1-7331E','D25-13305-2-7331E',
    'D25-13306-1-7331E','D25-13306-2-7331E',
    'D25-13307-1-7331E','D25-13307-2-7331E',
]
samp_CBE_EPO = [f'{x}_guide_split_BARCODES' for x in samp_CBE_EPO]
id_cbe = []
for x in ['input','d5','d15']:
    for y in range(5):
        for z in range(2):
            id_cbe.append(f'{x}-rep{y+1}-{z+1}')
id_cbe

samp_id_CBE_EPO= ['lib-1','lib-2'] + id_cbe + id_in_vivo

print(len(samp_CBE_EPO),len(samp_id_CBE_EPO))

62 62


In [13]:
# check if all the sequencing samples have been assigned to one of these four groups
len(set(samp_ABE_BC).union(set(samp_ABE_EPO)).union(set(samp_CBE_BC)).union(set(samp_CBE_EPO)))

188

## Dictionary of sample name and compiled crispresso data

In [14]:
#filepath for crispresso data
fp = '/Users/kexindong/Documents/GitHub/PhD-FSR-MH-Lab/07_B-ALL_resubmission_20250819/validation_screen_analysis/crispresso_data/original_unmerged'
edit_dict_ABE_BC = crispresso_compiler(samp_ABE_BC, samp_id_ABE_BC, fp)
edit_dict_ABE_EPO = crispresso_compiler(samp_ABE_EPO, samp_id_ABE_EPO, fp)
edit_dict_CBE_BC = crispresso_compiler(samp_CBE_BC, samp_id_CBE_BC, fp)
edit_dict_CBE_EPO = crispresso_compiler(samp_CBE_EPO, samp_id_CBE_EPO, fp)

In [21]:
LIB['classification'].unique()

array(['targeting guide', 'non-targeting control',
       'essential truncation guide', 'safe-targeting control'],
      dtype=object)

In [25]:
ABE_targ_list = LIB[(LIB['Editor'] == 'ABE') & (LIB['classification'].isin(['targeting guide', 'essential truncation guide']))]['gRNA_id'].to_list()

In [26]:
np.average(edit_dict_ABE_BC['bm1-1'][edit_dict_ABE_BC['bm1-1']['Guide_ID'].isin(ABE_targ_list)]['target_base_edit_perc'])

4.656308209956046

In [47]:
df = edit_dict_ABE_EPO['input-2'][
	(edit_dict_ABE_EPO['input-2']['Guide_ID'].isin(ABE_targ_list)) &
	(edit_dict_ABE_EPO['input-2']['Reads_aligned_all_amplicons'] >= 100)
]

In [48]:
np.average(df['target_base_edit_perc'])

1.269628299599965

## Merge NGS sequencing technical replicates

In [38]:
def MLE_merge_auto(edit_dict, samps_to_merge, new_name):
    """
    Combine replicates to generate best estimate of sensor editing.
    Automatically handles any number of replicates per group (≥1).
    """
    cols_to_add = [
        'Reads_in_input', 'Reads_lowqual', 'Reads_aligned_all_amplicons',
        'Reads_aligned_WT', 'Reads_aligned_HDR', 'Reads_unaligned', 'WT',
        'correct_edit', 'target_base_edit', 'byproduct_all',
        'byproduct_INDEL', 'byproduct_sub', 'byproduct_ambiguous',
    ]
    
    comb_holder = []

    for group in samps_to_merge:
        dfs = [edit_dict[sample] for sample in group]
        guide_check = [list(df['Guide_ID']) for df in dfs]
        assert all(g == guide_check[0] for g in guide_check), 'Guide_ID dont match'

        merged = dfs[0][cols_to_add].copy()
        for df in dfs[1:]:
            merged += df[cols_to_add]

        merged['Guide_ID'] = dfs[0]['Guide_ID']
        col_order = ['Guide_ID'] + cols_to_add
        out = merged[col_order]

        out['corr_perc'] = 100 * (out['correct_edit'] / out['Reads_aligned_all_amplicons'])
        out['target_base_edit_perc'] = 100 * (out['target_base_edit'] / out['Reads_aligned_all_amplicons'])
        out['WT_perc'] = 100 * (out['WT'] / out['Reads_aligned_all_amplicons'])
        out['byproduct_all_perc'] = 100 * (out['byproduct_all'] / out['Reads_aligned_all_amplicons'])
        out['byproduct_INDEL_perc'] = 100 * (out['byproduct_INDEL'] / out['Reads_aligned_all_amplicons'])
        out['byproduct_sub_perc'] = 100 * (out['byproduct_sub'] / out['Reads_aligned_all_amplicons'])
        out['byproduct_ambiguous_perc'] = 100 * (out['byproduct_ambiguous'] / out['Reads_aligned_all_amplicons'])
        out = out.fillna(0)
        out = out.sort_index()   # keep original row order
        comb_holder.append(out)

    return dict(zip(new_name, comb_holder))

In [ ]:
samp_id_ABE_BC

In [41]:
samps_to_merge = [
        [samp_id_CBE_BC[i], samp_id_CBE_BC[i+1]]
        for i in range(0, len(samp_id_CBE_BC), 2)
    ]
samp_id_CBE_BC

['lib-1',
 'lib-2',
 'input-rep1-1',
 'input-rep1-2',
 'input-rep2-1',
 'input-rep2-2',
 'input-rep3-1',
 'input-rep3-2',
 'input-rep4-1',
 'input-rep4-2',
 'input-rep5-1',
 'input-rep5-2',
 'd5-rep1-1',
 'd5-rep1-2',
 'd5-rep2-1',
 'd5-rep2-2',
 'd5-rep3-1',
 'd5-rep3-2',
 'd5-rep4-1',
 'd5-rep4-2',
 'd5-rep5-1',
 'd5-rep5-2',
 'd15-rep1-1',
 'd15-rep1-2',
 'd15-rep2-1',
 'd15-rep2-2',
 'd15-rep3-1',
 'd15-rep3-2',
 'd15-rep4-1',
 'd15-rep4-2',
 'd15-rep5-1',
 'd15-rep5-2',
 'spleen1-1',
 'spleen1-2',
 'bm1-1',
 'bm1-2',
 'men1-1',
 'men1-2',
 'spleen2-1',
 'spleen2-2',
 'bm2-1',
 'bm2-2',
 'men2-1',
 'men2-2',
 'spleen4-1',
 'spleen4-2',
 'bm4-1',
 'bm4-2',
 'men4-1',
 'men4-2']

In [52]:
new_sample_ids = list(dict.fromkeys(x.rsplit("-", 1)[0] for x in samp_id_CBE_BC))
new_sample_ids

['lib',
 'input-rep1',
 'input-rep2',
 'input-rep3',
 'input-rep4',
 'input-rep5',
 'd5-rep1',
 'd5-rep2',
 'd5-rep3',
 'd5-rep4',
 'd5-rep5',
 'd15-rep1',
 'd15-rep2',
 'd15-rep3',
 'd15-rep4',
 'd15-rep5',
 'spleen1',
 'bm1',
 'men1',
 'spleen2',
 'bm2',
 'men2',
 'spleen4',
 'bm4',
 'men4']

In [61]:
def merge_NGS_replicates(sample_ids, edit_dict): 
    samps_to_merge = [
        [sample_ids[i], sample_ids[i+1]]
        for i in range(0, len(sample_ids), 2)
    ]
    new_sample_ids = list(dict.fromkeys(x.rsplit("-", 1)[0] for x in sample_ids))
    comb_dict = MLE_merge_auto(edit_dict, samps_to_merge, new_sample_ids)
    return comb_dict

ABE_BC = merge_NGS_replicates(samp_id_ABE_BC, edit_dict_ABE_BC)
ABE_EPO = merge_NGS_replicates(samp_id_ABE_EPO, edit_dict_ABE_EPO)
CBE_BC = merge_NGS_replicates(samp_id_CBE_BC, edit_dict_CBE_BC)
CBE_EPO = merge_NGS_replicates(samp_id_CBE_EPO, edit_dict_CBE_EPO)

/var/folders/rl/65f1zcd575n092yn3xlqlqfw0000gn/T/ipykernel_28712/449173214.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out['corr_perc'] = 100 * (out['correct_edit'] / out['Reads_aligned_all_amplicons'])
/var/folders/rl/65f1zcd575n092yn3xlqlqfw0000gn/T/ipykernel_28712/449173214.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out['target_base_edit_perc'] = 100 * (out['target_base_edit'] / out['Reads_aligned_all_amplicons'])
/var/folders/rl/65f1zcd575n092yn3xlqlqfw0000gn/T/ipykernel_28712/4491732

In [62]:
ABE_EPO.keys()

dict_keys(['lib', 'input', 'd5', 'd15-rep1', 'd15-rep2', 'd15-rep3', 'd15-rep4', 'd15-rep5', 'spleen1', 'bm1', 'men1', 'spleen2', 'bm2', 'men2', 'spleen3', 'bm3', 'men3', 'spleen4', 'bm4', 'men4', 'spleen5', 'bm5', 'men5'])

In [64]:
df = ABE_EPO['input'][
	(ABE_EPO['input']['Guide_ID'].isin(ABE_targ_list)) &
	(ABE_EPO['input']['Reads_aligned_all_amplicons'] >= 100)
]

In [65]:
np.average(df['target_base_edit_perc'])

1.7064828782261923

In [66]:
def MLE_merge(edit_dict, samps_to_merge, new_name):
    """ 
    combine replicates to generate best estimate of sensor editing
    only works for combinations of 3 replicates (could be modified)
    """
    cols_to_add = ['Reads_in_input', 'Reads_lowqual',
       'Reads_aligned_all_amplicons', 'Reads_aligned_WT', 'Reads_aligned_HDR',
       'Reads_unaligned', 'WT', 'correct_edit', 'target_base_edit',
       'byproduct_all', 'byproduct_INDEL', 'byproduct_sub',
       'byproduct_ambiguous',]
    
    comb_holder = []
    for k in samps_to_merge:
        a = edit_dict[k[0]]
        b = edit_dict[k[1]]
        c = edit_dict[k[2]]

        #check that guide indeces match
        assert list(a['Guide_ID'])==list(b['Guide_ID'])==list(c ['Guide_ID']), 'indexes dont match'

        a2 = a[cols_to_add]
        b2 = b[cols_to_add]
        c2 = c[cols_to_add]

        comb = a2+b2+c2

        comb['Guide_ID'] = a['Guide_ID']

        #and reorganize columns
        col_order = ['Guide_ID'] + cols_to_add
        out = comb[col_order]

        #and then calcualte percentages
        out['corr_perc'] = 100*(out['correct_edit']/out['Reads_aligned_all_amplicons'])
        out['target_base_edit_perc'] = 100*(out['target_base_edit']/out['Reads_aligned_all_amplicons'])
        out['WT_perc'] = 100*(out['WT']/out['Reads_aligned_all_amplicons'])
        out['byproduct_all_perc'] =  100*(out['byproduct_all']/out['Reads_aligned_all_amplicons'])
        out['byproduct_INDEL_perc'] =  100*(out['byproduct_INDEL']/out['Reads_aligned_all_amplicons'])
        out['byproduct_sub_perc'] =  100*(out['byproduct_sub']/out['Reads_aligned_all_amplicons'])
        out['byproduct_ambiguous_perc'] =100*(out['byproduct_ambiguous']/out['Reads_aligned_all_amplicons'])
        out = out.fillna(0)

        comb_holder.append(out)

    out_dict = dict(zip(new_name, comb_holder))

    return out_dict

In [67]:
ABE_BC['bm1']

,Guide_ID,Reads_in_input,Reads_lowqual,Reads_aligned_all_amplicons,Reads_aligned_WT,Reads_aligned_HDR,Reads_unaligned,WT,correct_edit,target_base_edit,...,byproduct_INDEL,byproduct_sub,byproduct_ambiguous,corr_perc,target_base_edit_perc,WT_perc,byproduct_all_perc,byproduct_INDEL_perc,byproduct_sub_perc,byproduct_ambiguous_perc
0,gRNA_1000,4666,0,4666,4101,561,4,2378,0,561,...,0,2284,4,0.000000,12.023146,50.964423,49.035577,0.0,48.949850,0.085727
1,gRNA_1004,0,0,0,0,0,0,0,0,0,...,0,0,0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000
2,gRNA_10049,5983,0,5983,5954,23,6,3866,1,23,...,0,2110,6,0.016714,0.384423,64.616413,35.366873,0.0,35.266589,0.100284
3,gRNA_1006,1,0,1,1,0,0,1,0,0,...,0,0,0,0.000000,0.000000,100.000000,0.000000,0.0,0.000000,0.000000
4,gRNA_10106,631,0,631,574,56,1,545,0,56,...,0,85,1,0.000000,8.874802,86.370840,13.629160,0.0,13.470681,0.158479
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1628,gRNA_9736,0,0,0,0,0,0,0,0,0,...,0,0,0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000
1629,gRNA_9742,2,0,2,2,0,0,2,0,0,...,0,0,0,0.000000,0.000000,100.000000,0.000000,0.0,0.000000,0.000000
1630,gRNA_9745,7447,0,7447,7114,308,25,6455,3,308,...,0,964,25,0.040285,4.135894,86.679200,13.280516,0.0,12.944810,0.335706
1631,gRNA_990,1,0,1,1,0,0,1,0,0,...,0,0,0,0.000000,0.000000,100.000000,0.000000,0.0,0.000000,0.000000


In [68]:
fp = '/Users/kexindong/Documents/GitHub/PhD-FSR-MH-Lab/07_B-ALL_resubmission_20250819/validation_screen_analysis/crispresso_data/compact_lane_merged_unfiltered'
comb_dict = [ABE_BC, ABE_EPO, CBE_BC, CBE_EPO]
name_list = ['ABE_BC', 'ABE_EPO', 'CBE_BC','CBE_EPO']

for i in range(len(comb_dict)):
    single_dict = comb_dict[i]
    name = name_list[i]
    os.makedirs(os.path.join(fp, name), exist_ok=True)
    for sample_name, df in single_dict.items():
        filename = os.path.join(fp, name, f'{sample_name}_compact_unfiltered.csv')
        # print(filename)
        df.to_csv(filename, index=False)

In [71]:
# next time to use them:
fp = '/Users/kexindong/Documents/GitHub/PhD-FSR-MH-Lab/07_B-ALL_resubmission_20250819/validation_screen_analysis/crispresso_data/compact_lane_merged_unfiltered'

name_list = ['ABE_BC', 'ABE_EPO', 'CBE_BC', 'CBE_EPO']

loaded_dict = {}   # this will hold everything

for name in name_list:
    folder_path = os.path.join(fp, name)
    sample_files = sorted(os.listdir(folder_path))   # alphabetical order
    
    # dictionary for this screen
    screen_dict = {}

    for fname in sample_files:
        if fname.endswith('.csv'):
            sample_name = fname.replace('_compact_unfiltered.csv', '')
            fpath = os.path.join(folder_path, fname)
            screen_dict[sample_name] = pd.read_csv(fpath)

    loaded_dict[name] = screen_dict

# MLE

In [72]:
ABE_EPO_samps_to_merge = ABE_BC_samps_to_merge = [['bm1', 'bm2', 'bm3', 'bm4', 'bm5'], 
['d15-rep1', 'd15-rep2', 'd15-rep3','d15-rep4', 'd15-rep5'], 
['d5'],
['input'],
['lib'],
['men1', 'men2', 'men3','men4', 'men5'], 
['spleen1', 'spleen2', 'spleen3', 'spleen4', 'spleen5']]
ABE_EPO_new_name = ABE_BC_new_name = ['bm', 'd15', 'd5','input','lib','men', 'spleen']

CBE_BC_samps_to_merge = [['bm1', 'bm2', 'bm4'], 
['d15-rep1', 'd15-rep2', 'd15-rep3','d15-rep4', 'd15-rep5'], 
['d5-rep1', 'd5-rep2', 'd5-rep3','d5-rep4', 'd5-rep5'],
['input-rep1','input-rep2','input-rep3','input-rep4','input-rep5'],
['lib'],
['men1', 'men2', 'men4'], 
['spleen1', 'spleen2', 'spleen4']]
CBE_BC_new_name = ['bm', 'd15', 'd5','input','lib','men', 'spleen']

CBE_EPO_samps_to_merge = [['bm1', 'bm2', 'bm3', 'bm4', 'bm5'], 
['d15-rep1', 'd15-rep2', 'd15-rep3','d15-rep4', 'd15-rep5'], 
['d5-rep1', 'd5-rep2', 'd5-rep3','d5-rep4', 'd5-rep5'],
['input-rep1','input-rep2','input-rep3','input-rep4','input-rep5'],
['lib'],
['men1', 'men2', 'men3','men4', 'men5'], 
['spleen1', 'spleen2', 'spleen3', 'spleen4', 'spleen5']]
CBE_EPO_new_name = ['bm', 'd15', 'd5','input','lib','men', 'spleen']

In [73]:
MLE_ABE_BC = MLE_merge_auto(loaded_dict['ABE_BC'], ABE_BC_samps_to_merge, ABE_BC_new_name)
MLE_ABE_EPO = MLE_merge_auto(loaded_dict['ABE_EPO'], ABE_EPO_samps_to_merge, ABE_EPO_new_name)
MLE_CBE_BC = MLE_merge_auto(loaded_dict['CBE_BC'], CBE_BC_samps_to_merge, CBE_BC_new_name)
MLE_CBE_EPO = MLE_merge_auto(loaded_dict['CBE_EPO'], CBE_EPO_samps_to_merge, CBE_EPO_new_name)

/var/folders/rl/65f1zcd575n092yn3xlqlqfw0000gn/T/ipykernel_28712/449173214.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out['corr_perc'] = 100 * (out['correct_edit'] / out['Reads_aligned_all_amplicons'])
/var/folders/rl/65f1zcd575n092yn3xlqlqfw0000gn/T/ipykernel_28712/449173214.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out['target_base_edit_perc'] = 100 * (out['target_base_edit'] / out['Reads_aligned_all_amplicons'])
/var/folders/rl/65f1zcd575n092yn3xlqlqfw0000gn/T/ipykernel_28712/4491732

In [74]:
fp = '/Users/kexindong/Documents/GitHub/PhD-FSR-MH-Lab/07_B-ALL_resubmission_20250819/validation_screen_analysis/MLE'
comb_dict = [MLE_ABE_BC, MLE_ABE_EPO, MLE_CBE_BC, MLE_CBE_EPO]
name_list = ['ABE_BC', 'ABE_EPO', 'CBE_BC','CBE_EPO']

for i in range(len(comb_dict)):
    single_dict = comb_dict[i]
    name = name_list[i]
    os.makedirs(os.path.join(fp, name), exist_ok=True)
    for sample_name, df in single_dict.items():
        filename = os.path.join(fp, name, f'{sample_name}.csv')
        # print(filename)
        df.to_csv(filename, index=False)